**Create a fresh project and install the dependencies:**

mkdir llm-zoomcamp-hw2 && cd llm-zoomcamp-hw2       
uv init --no-workspace      
uv add onnxruntime tokenizers numpy tqdm minsearch gitsource        
uv add --dev huggingface-hub jupyter        
uv run python -m ipykernel install --user --name llm-zoomcamp-hw2 --display-name "llm-zoomcamp-hw2"     

**Need two helper scripts from the embed/ directory of the course repo:**

PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed     
wget $PREFIX/download.py        
wget $PREFIX/embedder.py

**By default download.py fetches Xenova/all-MiniLM-L6-v2, the ONNX version of the all-MiniLM-L6-v2 model from the lessons:**

uv run python download.py

## Q1. Embedding a query
Embed the following query:

*How does approximate nearest neighbor search work?*

The embedder returns a vector of 384 numbers. What's the first value (v[0])?

-0.31       
-0.02       
0.12        
0.44        

Q1. Solution:

In [22]:
from embedder import Embedder

embed = Embedder()

q1 = "How does approximate nearest neighbor search work?"

v = embed.encode(q1)
v[0]

np.float64(-0.02058203437252893)

## Loading the data

We pull the lesson pages from the course repository, the same way as in homework 1. We pin to commit 8c1834d so everyone works with the same data.

In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
documents[0]

## Q2. Cosine similarity

The embedder returns normalized vectors, so the dot product between two of them is their cosine similarity.

Take the page *02-vector-search/lessons/07-sqlitesearch-vector.md*, embed its content, and compute the cosine similarity with the query vector from Q1. What do you get?

0.07        
0.37        
0.68        
0.92

Q2. Solution:

In [8]:
# Find the specific document
doc = next(d for d in documents if d['filename'] == '02-vector-search/lessons/07-sqlitesearch-vector.md')

# Embed its content
v_doc = embed.encode(doc['content'])

# Cosine similarity (dot product of normalized vectors = cosine similarity)
similarity = v.dot(v_doc)
print(similarity)

0.36107026789538205


## Q3. Chunking and search by hand

A full page covers several topics, which waters down its embedding.

We chunk the pages the same way as in homework 1:

In [ ]:
# from gitsource import chunk_documents
# chunks = chunk_documents(documents, size=2000, step=1000)

We embed every chunk's content with encode_batch, stack the vectors into a matrix X, and score the Q1 query against all chunks:

In [ ]:
# scores = X.dot(v)

Which file does the highest-scoring chunk belong to (its filename)?

* 02-vector-search/lessons/03-embeddings-dataset.md     
* 02-vector-search/lessons/06-rag-vector.md     
* 02-vector-search/lessons/07-sqlitesearch-vector.md        
* 02-vector-search/lessons/09-onnx-embedder.md

Q3. Solution:

In [24]:
import numpy as np
from tqdm.auto import tqdm
from gitsource import chunk_documents

# 1. Chunk all documents
chunks = chunk_documents(documents, size=2000, step=1000)

# 2. Embed every chunk in batches
batch_size = 50
all_vectors = []

for i in tqdm(range(0, len(chunks), batch_size)):
    batch = chunks[i:i + batch_size]
    texts = [chunk['content'] for chunk in batch]
    batch_vectors = embed.encode_batch(texts)
    all_vectors.extend(batch_vectors)

# 3. Stack into a matrix
X = np.array(all_vectors)

# 4. Score against Q1 query vector
scores = X.dot(v)

# 5. Find the chunk
idx = np.argmax(scores)
print("Score:   ", scores[idx])
print("Filename:", chunks[idx]['filename'])
print("Content: ", chunks[idx]['content'][:200])  # first 200 chars as a preview

  0%|          | 0/6 [00:00<?, ?it/s]

Score:    0.6489016436447387
Filename: 02-vector-search/lessons/07-sqlitesearch-vector.md
Content:  rch. We score
the query against every document and pick the top ones. It always finds
the true top matches, but it pays for that by touching everything.

Approximate nearest neighbor (ANN) search take


## Q4. Vector search with minsearch

We've done vector search by hand, which is good for learning, but it's not what we do in practice. In practice we use libraries.

Let's use VectorSearch from minsearch and run a search for the following query:

*What metric do we use to evaluate a search engine?*

Which file is the filename of the first result?

* 02-vector-search/lessons/04-vector-search.md      
* 04-evaluation/lessons/05-search-metrics.md        
* 04-evaluation/lessons/13-llm-as-judge.md      
* 05-monitoring/lessons/04-metrics.md       

Q4. Solution:

In [26]:
from tqdm.auto import tqdm
from gitsource import chunk_documents

# 1. Chunk all documents
chunks = chunk_documents(documents, size=2000, step=1000)

# 2. Embed every chunk in batches
batch_size = 50
all_vectors = []

for i in tqdm(range(0, len(chunks), batch_size)):
    batch = chunks[i:i + batch_size]
    texts = [chunk['content'] for chunk in batch]
    batch_vectors = embed.encode_batch(texts)
    all_vectors.extend(batch_vectors)

# 3. Turn them into 2 dimensional array(matrix) where rows are documents(vectors) and columns are dimensions of the vectors
import numpy as np
X = np.array(all_vectors)

# 4. Index our documents and vectors
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["content"])
vindex.fit(X, chunks)

# 5. Searching
from embedder import Embedder
embed = Embedder()

query = "What metric do we use to evaluate a search engine?"

query_vector = embed.encode(query)

results = vindex.search(query_vector, num_results=5)
# results

[result['filename'] for result in results]

  0%|          | 0/6 [00:00<?, ?it/s]

['04-evaluation/lessons/05-search-metrics.md',
 '04-evaluation/lessons/01-intro.md',
 '01-agentic-rag/lessons/05-search.md',
 '04-evaluation/lessons/01-intro.md',
 '04-evaluation/lessons/15-next-steps.md']

## Q5. Text search vs vector search

Vector search matches by meaning, keyword search by exact words.

Let's compare them. Index the same chunks with Index from minsearch. Use content as a text field.

Run both searches for this query:

*How do I store vectors in PostgreSQL?*

Take the top 5 results from each method. Which file shows up in the vector results but not in the text results?

* 02-vector-search/lessons/01-intro.md      
* 02-vector-search/lessons/02-embeddings.md     
* 02-vector-search/lessons/08-pgvector.md       
* 03-orchestration/lessons/05-rag.md

Q5. Solution:

In [ ]:
# Ingestion
from gitsource import GithubRepositoryDataReader, chunk_documents

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

# Chunk all documents
chunks = chunk_documents(documents, size=2000, step=1000)

# User's FAQ question
question = "How do I store vectors in PostgreSQL?"

In [34]:
# For question 6:
question = "How do I give the model access to tools?"

## Text Search

In [35]:
# 1. Index with minisearch
from minsearch import Index

index = Index(
    text_fields=["content"]
)

index.fit(chunks)

# 2. Search with text search
text_results = index.search(
    question,
    num_results=5
)

# text_results

[result['filename'] for result in text_results]

['01-agentic-rag/lessons/14-agentic-loop.md',
 '01-agentic-rag/lessons/13-function-calling.md',
 '01-agentic-rag/lessons/13-function-calling.md',
 '01-agentic-rag/lessons/13-function-calling.md',
 '04-evaluation/lessons/02-ground-truth.md']

## Vector Search

In [36]:
from tqdm.auto import tqdm
from embedder import Embedder

embed = Embedder()

# 1. Embed every chunk in batches
batch_size = 50
all_vectors = []

for i in tqdm(range(0, len(chunks), batch_size)):
    batch = chunks[i:i + batch_size]
    texts = [chunk['content'] for chunk in batch]
    batch_vectors = embed.encode_batch(texts)
    all_vectors.extend(batch_vectors)

# 2. Turn all vectors into 2 dimensional array(matrix) where rows are documents(vectors) and columns are dimensions of the vectors
import numpy as np
X = np.array(all_vectors)

# 3. Index our documents and vectors
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["content"])
vindex.fit(X, chunks)

# 4. Vector search
from embedder import Embedder
embed = Embedder()

query_vector = embed.encode(question)

vector_results = vindex.search(query_vector, num_results=5)
# vector_results

[result['filename'] for result in vector_results]

  0%|          | 0/6 [00:00<?, ?it/s]

['01-agentic-rag/lessons/01-intro.md',
 '04-evaluation/lessons/02-ground-truth.md',
 '01-agentic-rag/lessons/16-other-frameworks.md',
 '01-agentic-rag/lessons/15-frameworks.md',
 '01-agentic-rag/lessons/13-function-calling.md']

## Q6. Hybrid search

Both vector and text search have their strengths and weaknesses. Vector search matches by meaning, so it finds relevant pages even when they use words different from the query. But it can miss exact terms like names, codes, or rare keywords. Text search is the opposite: it nails exact words but misses paraphrases and synonyms.

We don't have to pick one or the other - we can use both and merge their results. This approach is called "hybrid search".

Each search produces its own ranked list, so we need a way to combine them into one. In this homework we use Reciprocal Rank Fusion (RRF). It ignores the raw scores from each method, which live on different scales and aren't directly comparable. Instead, it looks only at the position of each document in each list.

Every document scores by its position (rank, starting at 0) in each list, and we sum the scores across lists with a constant k = 60:

In [ ]:
# RRF(d) = sum over lists of  1 / (k + rank(d))

"Sum over lists" means we go through every ranked list and, for each list where the document appears, add its *1 / (k + rank)* contribution. A document found by both searches collects a score from each list, while one found by only a single search collects just one.

The constant k controls how much the exact rank matters. A larger k flattens the gap between positions, so the difference between rank 0 and rank 5 counts for less. A smaller k does the opposite: it sharpens that gap, so being at the top of a list matters much more.

The value 60 comes from the original RRF paper and is the usual default. You rarely need to tune it. Lower it when only the top results matter. Raise it to reward documents that appear across many lists, even when they never quite reach the top.

A document that ranks well in both lists ends up higher than one that's only strong in a single list.

In [37]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

Now run the query *"How do I give the model access to tools?"* with vector and text search and fuse the results with rrf:

In [ ]:
# results = rrf([vector_results, text_results])

Which file is ranked first after RRF?

* 01-agentic-rag/lessons/01-intro.md        
* 01-agentic-rag/lessons/13-function-calling.md     
* 01-agentic-rag/lessons/14-agentic-loop.md     
* 01-agentic-rag/lessons/16-other-frameworks.md     

Notice that this file isn't first in either search on its own - it wins because it ranks high in both.

## Q6. Solution:

In [38]:
results = rrf([vector_results, text_results])
# results

[result['filename'] for result in results]

['01-agentic-rag/lessons/13-function-calling.md',
 '01-agentic-rag/lessons/01-intro.md',
 '01-agentic-rag/lessons/14-agentic-loop.md',
 '04-evaluation/lessons/02-ground-truth.md',
 '01-agentic-rag/lessons/16-other-frameworks.md']